# 🌿 Crop Disease Detection from Smartphone Photos
### Machine Learning & Preprocessing Experimentation Notebook

Welcome! This notebook walks you through the core AI concepts behind the **Crop Disease Detector** project. We will:
1. **Load and preprocess** plant leaf photos.
2. **Demonstrate data augmentation** (rotation, zoom, flips) to make the model robust.
3. **Construct the Transfer Learning network** based on Google's pre-trained **MobileNetV2** CNN.
4. **Explore model optimization** using post-training 8-bit quantization for lightweight deployment.

## 1. Environment Setup & Dependencies
First, let's verify that we have all the key libraries installed and load them into our active python session.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

print(f"TensorFlow Version: {tf.__version__}")
print("All core dependencies loaded successfully!")

## 2. Locate and Inspect Dataset
We use the **PlantVillage** crop disease dataset from Kaggle. Our helper module scans the files recursively to find the image folder. Let's load the paths.

In [ ]:
from utils.helpers import create_folders, find_dataset_images_root

# Ensure standard folders exist
create_folders()

# Locate dataset root
dataset_root = find_dataset_images_root("dataset")
print(f"Active dataset folder: {dataset_root}")
if os.path.exists(dataset_root) and len(os.listdir(dataset_root)) > 0:
    classes = [d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))]
    print(f"Found {len(classes)} crop classes. Example class names: {classes[:3]}")
else:
    print("ℹ️ Note: If you haven't run train.py or set up kaggle.json, the dataset folder will be empty.")

## 3. Data Augmentation Preview
Data augmentation creates varied copies of our training images by applying random rotations, flips, zooms, and contrast shifts. This prevents the CNN from memorizing exact leaf rotations and helps it generalize to real-world smartphone photos taken from different angles and lighting.

Let's load one of our beautiful generated sample photos from `assets/sample_images` and visualize it before and after entering the data augmentation pipeline.

In [ ]:
from utils.preprocess import get_data_augmentation_pipeline

# Load a sample leaf photograph
sample_image_path = os.path.join("..", "assets", "sample_images", "Tomato_Early_Blight.jpg")
if not os.path.exists(sample_image_path):
    # Fallback in case notebook is run from root directory
    sample_image_path = os.path.join("assets", "sample_images", "Tomato_Early_Blight.jpg")

if os.path.exists(sample_image_path):
    img = Image.open(sample_image_path).convert('RGB')
    img_resized = img.resize((224, 224))
    img_array = np.array(img_resized, dtype=np.float32)
    
    # Create a batch channel: (1, 224, 224, 3) as Keras layers expect batches
    img_batched = np.expand_dims(img_array, axis=0)
    
    # Instantiate our data augmentation pipeline
    augmentation = get_data_augmentation_pipeline()
    
    # Plot the original image and 4 random augmentations side-by-side
    plt.figure(figsize=(15, 6))
    
    # Original
    plt.subplot(1, 5, 1)
    plt.imshow(img_array.astype(np.uint8))
    plt.title("Original Leaf")
    plt.axis("off")
    
    # Augmentations
    for i in range(4):
        augmented_tensor = augmentation(img_batched, training=True) # training=True activates random actions
        augmented_img = augmented_tensor[0].numpy().astype(np.uint8)
        
        plt.subplot(1, 5, i + 2)
        plt.imshow(augmented_img)
        plt.title(f"Augmented Variation {i+1}")
        plt.axis("off")
        
    plt.tight_layout()
    plt.show()
else:
    print("Error: Sample image not found! Make sure you created sample files first.")

## 4. Building the MobileNetV2 Transfer Learning Architecture
Now we'll build our neural net structure. We start with Google's **MobileNetV2** base trained on **ImageNet**, freeze its pre-learned weight layers, and add a custom global average pooling, dropout, and a final dense classification layer.

In [ ]:
def create_experiment_model(num_classes=15):
    # 1. Native data augmentation sequential layer
    augmentation = get_data_augmentation_pipeline()
    
    # 2. Input layer (takes raw float array [0, 255])
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = augmentation(inputs)
    
    # 3. Scale range from [0, 255] -> [-1, 1] for MobileNetV2 compatibility
    x = tf.keras.layers.Rescaling(scale=1./127.5, offset=-1.0)(x)
    
    # 4. MobileNetV2 backbone (headless feature extractor)
    base = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False # Freeze layers!
    
    # Pass tensors through backbone (training=False preserves batchnorm weights)
    x = base(x, training=False)
    
    # 5. Classifier Head
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    
    # Final combined Keras model
    model = tf.keras.Model(inputs, outputs, name="MobileNetV2_Transfer_Learning")
    return model

exp_model = create_experiment_model()
exp_model.summary()

## 5. Model Quantization and Edge Compression
Once training completes and our model accuracy is high, we compress it using **TensorFlow Lite (TFLite)**.
By applying **Post-Training Quantization**, we map all 32-bit floating-point weights ($32$-bit floats) down to compressed $8$-bit floats/integers. This:
- **Reduces storage footprint** from ~15 MB down to ~4 MB.
- **Increases inference speeds** significantly, allowing the CPU to execute math calculations faster with zero dependencies on external GPU resources.

Below is the code template we run at the end of training to export our snappy optimized TFLite binary:

In [ ]:
print("TFLite Quantization Export Code Template:")
print("""
converter = tf.lite.TFLiteConverter.from_keras_model(trained_h5_model)
# Enable default post-training quantization optimizations
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('model/crop_disease_model.tflite', 'wb') as f:
    f.write(tflite_model)
""")

---
### 🎉 Congratulations!
You have explored the complete machine learning architecture of LeafShield AI! Feel free to run this notebook in your local Python/Jupyter environment after installing the libraries in `requirements.txt`.